#### PIP

In [74]:
%pip install -q nltk
%pip install -q spacy


[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Imports and Downloads

In [75]:
import nltk

nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('universal_tagset', quiet=True)

True

In [76]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [77]:
import spacy
from spacy.tokens import Doc

# Load the spaCy model globally (en_core_web_sm is lightweight and efficient)
nlp = spacy.load("en_core_web_sm")

#### Initalizing Constants

In [176]:
# 9 Classification Labels
LABELS = [
    'name_calling,labeling',
    'repetition',
    'causal_oversimplification',
    'doubt',
    'loaded_language',
    'appeal_to_fear_prejudice',
    'flag_waving',
    'exaggeration,minimisation',
    'not_propaganda'
]


# POS UNIVERSAL TAGSET
UNIVERSAL_TAGSET = [
    "ADJ","ADP","ADV","CONJ","DET","NOUN","NUM","PRT","PRON","VERB",".","X"
]


# CUSTOM TRUNCATED NER TAGSET
NER_TAG = [
    'PERSON','ORG','GPE','DATE','NORP','CARDINAL','ORDINAL','TIME','LOC', 'O', 
    # 'MONEY','EVENT','PERCENT','WORK_OF_ART','FAC','LAW','PRODUCT','LANGUAGE','QUANTITY',
    ]

# CUSTOM STOPWORDS
CUSTOM_STOPWORDS = ["the" , ",", "to", "of", "and", "in", "a", "that"]

#### Handling Raw Input

This contains the function(s) that work directly on the source string data itself and therefore pertain to all subsequent methodologies.

In [ ]:
import re

def universal_cleaning(raw_text: str) -> str:
    
    # Clear leading/trailing whitespace
    text = raw_text.strip() 
    
    # Strip out python escape backslashes
    text = text.replace("\\'", "'").replace('\\"', '"')
    
    # Standardize quotes to flat quotes
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    
    # Collapse specific apostrophes: won't -> wont, lukes' -> lukes
    text = re.sub(r"(?<=\w)'(?=\w)|(?<=[sS])'", '', text)
    
    # Remove/Replace artifacts with a space, Matches: \ / [ ] * | @ space - . : $ # + =
    artifact_pattern = r'[\\/\[\]*|@\ \-.:$#+=]'
    text = re.sub(artifact_pattern, ' ', text)
    
    # Enforce strict padding boundaries around structural tags
    text = text.replace("<BOS>", " <BOS> ").replace("<EOS>", " <EOS> ")
    
    # 6. Collapse any accidental double spaces
    text = " ".join(text.split())
    
    return text

In [ ]:
def process_row(row):
    label = row['label']
    raw_text = row['tagged_in_context']
    
    processed_text = universal_cleaning(raw_text)

    return label, processed_text

#### Whole Word Tokenization

In [ ]:
import re

def tokenize_whole_words(text: str) -> list[str]:
    """
    Whole-word tokenizer.
    Replaces numbers with 'num'.
    retains internal word punctuation, and keeps individual punctuation marks.
    Case normalise
    """

    # Number mapper
    localized_text = re.sub(r'\b\d+(?:,\d+)*\b', 'num', text)
    
    # Regex Tokenizer
    pattern = r"<BOS>|<EOS>|(?:[a-zA-Z]\.)+|[a-zA-Z0-9]+(?:[-']?[a-zA-Z0-9]+)*|[^\w\s]"
    raw_tokens = re.findall(pattern, localized_text)
    
    # Case normalise
    clean_tokens = [
        token if token in ["<BOS>", "<EOS>"] else token.lower()
        for token in raw_tokens
    ]
    
    return clean_tokens

#### POS Tagging

In [177]:
from nltk.tag.perceptron import PerceptronTagger
from nltk.tag import map_tag

tagger = PerceptronTagger()

def tag_pos_pipeline(text: str, tagset: list) -> list[tuple[str, str]]:
    """
    Tag tokens with their POS.
    POS tags are mapped from PennTree to Universal Set
    returns a list of tags only. 
    """

    tokens = tokenize_whole_words(text) # string to tokens
    raw_tags = tagger.tag(tokens) # tokens to tags
    
    return [
        ("__BOUNDARY__") if token == "<BOS>" else
        ("__BOUNDARY__") if token == "<EOS>" else
        ("NUM") if token.lower() == "num" else # capture num rule from string formatting
        (".") if token in ['"', "'", '`'] else # override rules
        (map_tag('en-ptb', 'universal', tag))
        for token,tag in raw_tags
    ]

#### NER Tagging

In [178]:
def tag_ner_pipeline(text: str, tagset: list) -> list[tuple[str, str]]:
    """
    Tag tokens with their NER.
    Less freq NER tags are mapped to MISC
    returns a list of tags only. 
    """

    ALLOWED_TAGS = set(tagset) # custom reduced tag set

    tokens = tokenize_whole_words(text) # string to tokens
    
    # tagger
    doc = Doc(nlp.vocab, words=tokens)
    for name, proc in nlp.pipeline:
        doc = proc(doc)
        
    ner_tags = []
    for token in doc:

        if token.text in ["<BOS>", "<EOS>"]:
            ner_tags.append("__BOUNDARY__")
            
        elif token.ent_type_: # tagged
            full_tag = f"{token.ent_type_}"
            
            if token.ent_type_ in ALLOWED_TAGS:
                ner_tags.append(full_tag)
            else:
                ner_tags.append(f"MISC") # collapse less freq tags into misc 
                
        else:
            ner_tags.append("O")
            
    return ner_tags

#### Vocabulary Construction

In [ ]:
from collections import Counter

def get_hapax_legomena(counter: Counter) -> list[str]:
    """
    Returns a list of all words that appear exactly once in the corpus.
    """
    return [word for word, count in counter.items() if count == 1]


In [ ]:
def get_vocab_list(counter: Counter) -> list[str]:
    """
    Returns a list of all words that appear more than once in the corpus.
    """
    return ["__UNK__"] + [word for word, count in counter.items() if count > 1]

In [ ]:
def build_vocab_index(vocab_list: list[str], stop_set: list) -> dict[str, int]:
    """
    Takes a list of words and returns a dictionary mapping each word to its unique index position.
    """
    v = {word: i for i, word in enumerate(word for word in vocab_list if word not in stop_set + ["<EOS>","<BOS>"])}
    return v

In [157]:
def build_tag_index(tagset: list[str]) -> dict[str, int]:
    """
    Creates a mapping of tags to index positions.
    """
    return {tag: i for i, tag in enumerate(tagset) if tag not in ["O", "__BOUNDARY__"]}

The code below processes each row in the training data by tokenizing, POS and NER tagging the string input. With this, global counters are initalized to compute the total corpus values for tokens and tagset.

In [252]:
import csv
from collections import Counter

file_path = '../data/propaganda_train.tsv'

global_vocab_counts = Counter()
global_pos_counts = Counter()
global_ner_counts = Counter()


with open(file_path, mode='r', encoding='utf-8') as file:
    tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
    
    for row_idx, raw_row in enumerate(tsv_reader, start=1):
        
        label, text = process_row(raw_row) # STRING FORMATTED TEXT
        
        if text.count("<BOS>") != 1 or text.count("<EOS>") != 1:
            print(f"WARNING: Row {row_idx}, Invalid tag format (Missing or Duplicate tags)")
            continue

        if label not in LABELS:
            print(f"WARNING: Row {row_idx}, Invalid label")
            continue

        tokenized = tokenize_whole_words(text)
        pos = tag_pos_pipeline(text, UNIVERSAL_TAGSET)
        ner = tag_ner_pipeline(text, NER_TAG)

        if len(tokenized) != len(pos) != len(ner):
            print("Sequences are not the same length")

        global_vocab_counts.update(tokenized)
        global_pos_counts.update(pos)
        global_ner_counts.update(ner)

In [260]:
print("Vocab Counts:", str(global_vocab_counts))
print("POS Counts:  ", str(global_pos_counts))
print("NER Counts:  ", str(global_ner_counts))

Vocab Counts: Counter({'the': 4287, ',': 3279, '<BOS>': 2560, '<EOS>': 2560, 'to': 2114, 'of': 1965, 'and': 1689, '"': 1643, 'in': 1285, 'a': 1230, 'that': 1092, 'is': 822, 'num': 653, 'for': 534, 'it': 484, 'on': 463, 'with': 453, 'as': 443, 'was': 436, 'this': 411, 'he': 389, 'be': 361, 'have': 358, 'are': 356, 'not': 342, 'by': 332, 'his': 318, 'they': 300, 'from': 296, 'who': 288, 'has': 285, 'at': 270, 'i': 256, 'an': 254, 'but': 230, 'we': 214, 'said': 211, 'will': 207, '?': 191, 'our': 188, 'their': 187, 'you': 182, 'about': 181, 'one': 179, 'all': 175, 'people': 162, 'or': 162, 'which': 159, 'its': 156, 'trump': 150, '(': 148, 'out': 147, 'were': 144, ')': 144, 'been': 143, 'if': 143, 'no': 140, 'so': 139, 'what': 138, 'do': 137, 'would': 134, 'when': 125, 'up': 125, 'had': 125, '—': 125, 'church': 123, "'": 121, '–': 116, 'she': 113, 'them': 113, 'now': 110, 'her': 110, 'him': 108, 'more': 107, 'against': 107, 's': 106, 'after': 106, 'because': 103, 'there': 103, 'any': 99, 'a

Using the global counters we can construct vocabulary lists. Using this `vocab_list` and the pre-initalized POS and NER tag lists, dictionaries are constructed turn these lists into the tools for constructing sparse vectors. 

The dictionary keys are the terms or tags and the value is their index from the respective list. 

```
{
    "token/tag_1":0,
    "token/tag_1":1,

}
```

In [229]:
import csv

file_path = '../data/silver_train.tsv'

global_vocab_counts_silver = global_vocab_counts.copy()

with open(file_path, mode='r', encoding='utf-8') as file:
    tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
    
    for row_idx, raw_row in enumerate(tsv_reader, start=1):
        
        label, text = process_row(raw_row) # STRING FORMATTED TEXT
        
        if text.count("<BOS>") != 1 or text.count("<EOS>") != 1:
            print(f"WARNING: Row {row_idx}, Invalid tag format (Missing or Duplicate tags)")
            continue

        if label not in LABELS:
            print(f"WARNING: Row {row_idx}, Invalid label")
            continue

        tokenized = tokenize_whole_words(text)
        
        in_snippet = False
        for token in tokenized:
            if token == "<BOS>":
                in_snippet = True
                continue
            elif token == "<EOS>":
                in_snippet = False
                continue
            
            if in_snippet:
                if token in global_vocab_counts:
                    global_vocab_counts_silver[token] += 1



gold_singletons = {word for word, count in global_vocab_counts.items() if count == 1}
silver_singletons = {word for word, count in global_vocab_counts_silver.items() if count == 1}
rescued_singletons = gold_singletons - silver_singletons


print(f"Gold Singleons: {len(gold_singletons)}")
print(f"Silver Singleons: {len(silver_singletons)}")
print(f"Total Singletons Rescued from __UNK__ Status: {len(rescued_singletons)}")
print("\nSample of Rescued Lexical Triggers:")
print(list(rescued_singletons)[:20])

Gold Singleons: 4037
Silver Singleons: 3409
Total Singletons Rescued from __UNK__ Status: 628

Sample of Rescued Lexical Triggers:
['engineered', 'reminder', 'determination', 'wilt', 'biases', 'introducing', 'heroes', 'restrict', 'populace', 'require', 'clueless', 'centrifuges', 'harder', 'bergs', 'glory', 'stem', 'enforcers', 'aspects', 'vibrant', 'smokescreen']


In [248]:
hapax_words_list = get_hapax_legomena(global_vocab_counts)
hapax_words_list_silver = get_hapax_legomena(global_vocab_counts_silver)

vocab_list = get_vocab_list(global_vocab_counts)
vocab_list_silver = get_vocab_list(global_vocab_counts_silver)

word_to_index = build_vocab_index(vocab_list, CUSTOM_STOPWORDS)
word_to_index_silver = build_vocab_index(vocab_list_silver, CUSTOM_STOPWORDS)

pos_to_index = build_tag_index(UNIVERSAL_TAGSET)
ner_to_index = build_tag_index(["MISC"] + NER_TAG)

In [250]:
print(f"Training Corpus Vocab including __UNK__: {len(word_to_index)}")
print(f"Training Corpus Vocab (Silver Enhanced) including __UNK__: {len(word_to_index_silver)}")
print(f"")
print(f"Both with a custom stopword removal: {CUSTOM_STOPWORDS}")
print(f"")
print(f"POS Vector Slots {len(pos_to_index)}: {list(pos_to_index.keys())}")
print(f"NER Vector Slots {len(ner_to_index)}: {list(ner_to_index.keys())}")
print(f"")

Training Corpus Vocab including __UNK__: 4618
Training Corpus Vocab (Silver Enhanced) including __UNK__: 5246

Both with a custom stopword removal: ['the', ',', 'to', 'of', 'and', 'in', 'a', 'that']

POS Vector Slots 12: ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRT', 'PRON', 'VERB', '.', 'X']
NER Vector Slots 10: ['MISC', 'PERSON', 'ORG', 'GPE', 'DATE', 'NORP', 'CARDINAL', 'ORDINAL', 'TIME', 'LOC']



#### Standardize MLP Head

In [306]:
import torch
import torch.nn as nn

class StandardizedClassificationHead(nn.Module):
    """
    MLP Head optimized for functional, row-by-row single instance streaming.
    Swaps BatchNorm1d with LayerNorm to handle batch sizes of 1 safely.
    """
    def __init__(self, input_dim: int, hidden_dim: int = 64, num_classes: int = 9, dropout_p: float = 0.3):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),  # Safe for batch_size=1; replaces BatchNorm1d
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

#### BoW Vectors

In [268]:
def string_to_bow_vector(string, vocab):
    tokenized = tokenize_whole_words(string)
    sequence_vector = [0] * len(vocab)

    for token in tokenized:
        if token in ["<EOS>", "<BOS>"]:
            continue 
        elif token in CUSTOM_STOPWORDS:
            continue
        elif token in hapax_words_list:
            sequence_vector[vocab["__UNK__"]] += 1
        else: 
            sequence_vector[vocab[token]] += 1
    
    return sequence_vector

In [269]:
def tagset_to_vector(string, tagset, tag_type):

    if tag_type == "POS":
        tag_list = tag_pos_pipeline(string, UNIVERSAL_TAGSET)
    elif tag_type == "NER":
        tag_list = tag_ner_pipeline(string, NER_TAG)
    else:
        print("tag type error")

    # INITALISE EMPTY TAG VECTOR
    sequence_vector = [0] * len(tagset)

    # POPULATE EMPTY VECTOR
    for tag in tag_list:
        if tag == "__BOUNDARY__":
            continue
        elif tag == "O":
            continue
        else:
           sequence_vector[tagset[tag]] += 1    # increment tag index

    return  sequence_vector

In [286]:
import torch
import torch.nn as nn

def string_to_input_vector(string_text, vocab_index, pos_index, ner_index):

    # TODO: extract bound indices and route IF vectors to snippet or full

    vocab_vector = string_to_bow_vector(string_text, vocab_index)
    pos_vector = tagset_to_vector(string_text, pos_index, "POS")
    ner_vector = tagset_to_vector(string_text, ner_index, "NER")

    # Convert your manual feature counts into discrete floating-point tensors
    t_vocab = torch.tensor(vocab_vector, dtype=torch.float32)
    t_pos = torch.tensor(pos_vector, dtype=torch.float32)
    t_ner = torch.tensor(ner_vector, dtype=torch.float32)

    # # Concat along dimension 0 to build your structural vector tuple [x_sem || x_pos || x_ner]
    x_combined = torch.cat([t_vocab, t_pos, t_ner], dim=0)
        
    # # Add the mandatory PyTorch batch dimension: shape shifts from (input_dimension) to (1, input_dimension)
    x_batched = x_combined.unsqueeze(0)

    return x_batched

In [287]:
import csv

file_path = '../data/train_20.tsv'

with open(file_path, mode='r', encoding='utf-8') as file:
    tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
    
    for row_idx, raw_row in enumerate(tsv_reader, start=1):
        
        label, text = process_row(raw_row) # STRING FORMATTED TEXT

        input_vector = string_to_input_vector(text, word_to_index, pos_to_index, ner_to_index)
        input_vector_silver = string_to_input_vector(text, word_to_index_silver, pos_to_index, ner_to_index)

print("Input Vector Example:", input_vector)
print("Dims of Vector:", len(input_vector[0]))

print("Input Vector Example:", input_vector_silver)
print("Dims of Vector:", len(input_vector_silver[0]))

Input Vector Example: tensor([[3., 0., 0.,  ..., 0., 0., 0.]])
Dims of Vector: 4640
Input Vector Example: tensor([[3., 0., 0.,  ..., 0., 0., 0.]])
Dims of Vector: 5268


#### Forward Pass (MLP Head)

In [288]:
def head_input_dim(vocab: dict, pos: dict, ner: dict):
    """ 
    Compute the method head input dims dynamically based on the vocab and tagset input vectors
    """
    d_vocab = len(vocab)
    d_pos = len(pos)
    d_ner = len(ner)
    return d_vocab + d_pos + d_ner

In [298]:
import csv
import torch
import torch.nn as nn

file_path = '../data/train_20.tsv'

input_dimension = head_input_dim(word_to_index, pos_to_index, ner_to_index)

model = StandardizedClassificationHead(input_dim=input_dimension, num_classes=len(LABELS))
model.eval() # Freeze Dropout and LayerNorm parameters

with open(file_path, mode='r', encoding='utf-8') as file:
    tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
    
    for row_idx, raw_row in enumerate(tsv_reader, start=1):
        
        label, text = process_row(raw_row) # STRING FORMATTED TEXT

        input_vector = string_to_input_vector(text, word_to_index, pos_to_index, ner_to_index)
        input_vector_silver = string_to_input_vector(text, word_to_index_silver, pos_to_index, ner_to_index)

        with torch.no_grad():
            logits = model(input_vector)
            predicted_idx = torch.argmax(logits, dim=1).item()

print(f"""
      Processed Sample {row_idx} 
      Tensor Shape: {input_vector.shape}
      Predicted Class Index: {predicted_idx}
      Actual Class Index: {label}
""")


      Processed Sample 20 
      Tensor Shape: torch.Size([1, 4640])
      Predicted Class Index: 1
      Actual Class Index: causal_oversimplification



#### Backward Training Pass

- lr 0.001 to 0.005 as was aggressive overfitting. find gentle more stable min
- hidden hims 128 to 64 as larger was overpowing the small samples space.
- droptout 0.3 to 0.5 as dev loss climbing implying co-adpatioation of aprams. dynamically uncouple these connections and force the network to rely on more varied features.

regularizers:
- dropout_p: functions at the network-layer level during training passes (model.train()) by randomly forcing a specific percentage (currently 30% or 0.3) of hidden layer activations to zero. This mechanism acts as a powerful regularizer because it prevents hidden nodes from co-adapting too tightly to specific training shortcuts—such as a single high-frequency keyword trigger—forcing the MLP to distribute its learning across more robust, generalized pathways.
- weight_decay: It is the mathematical implementation of L2 Regularization. Weight decay adds a penalty proportional to the square of your parameter magnitudes directly to the loss function. This penalizes the model for allowing any single weight connection to grow excessively large or dominant, keeping the network weights small, smooth, and less sensitive to minor variations in input data.
- adam: The "W" in AdamW explicitly stands for Decoupled Weight Decay. In the older, standard Adam optimizer, L2 weight decay was mathematically mixed into the adaptive moving averages of the gradients. This combination accidentally dampened the regularization effect when individual parameter learning rates scaled up or down. AdamW solves this by decoupling the regularizer entirely: it subtracts a small fraction of the weight at each step completely independent of the gradient history. Therefore, weight_decay is AdamW's native, highly optimized regularizer.


In [ ]:
def trainer(DATASET_FILE, LABELS, MODEL, EPOCHS, OPTIMIZER):
    label_idx = {label: i for i, label in enumerate(LABELS)} 
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        print(f"--- Starting Epoch {epoch} ---")

        running_train_loss = 0.0
        train_samples = 0
        
        running_val_loss = 0.0
        val_samples = 0

        with open(DATASET_FILE, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            
            for row_idx, raw_row in enumerate(tsv_reader, start=1):
                label, text = process_row(raw_row)

                # TODO: set up type router: bow, word2vec, deberta
                input_vector = string_to_input_vector(text, word_to_index, pos_to_index, ner_to_index)
                # TODO: function is calling global variables within function: word_to_index, pos_to_index, ner_to_index
                # TODO: still would be solved with a class where vocab(s) inited
            
                x_batched = input_vector
                y_target = torch.tensor([label_idx[label]], dtype=torch.long)
                
                if row_idx % 10 == 0: # 10% of data routed to validation (Modulo Split)
                    MODEL.eval()  # Freeze layers
                    with torch.no_grad():
                        logits = MODEL(x_batched)
                        val_loss = criterion(logits, y_target)

                        running_val_loss += val_loss.item()
                        val_samples += 1
                else:
                    MODEL.train()
                    OPTIMIZER.zero_grad()
                    logits = MODEL(x_batched)
                    loss = criterion(logits, y_target)
                    loss.backward()
                    OPTIMIZER.step()

                    running_train_loss += loss.item()
                    train_samples += 1

        epoch_train_loss = running_train_loss / train_samples if train_samples > 0 else 0.0
        epoch_val_loss = running_val_loss / val_samples if val_samples > 0 else 0.0
        
        print(f"Epoch {epoch} Results | Avg Train Loss: {epoch_train_loss:.4f} | Avg Dev Loss: {epoch_val_loss:.4f}\n")

        # DYNAMIC EARLY STOPPING SAVE CHECK
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            
            # save parameters (weights)
            torch.save(MODEL.state_dict(), 'propaganda_mlp_weights.pt')  # [cite: 1]
            print(f"--> New best validation loss achieved, weights saved.\n")
        else:
            print(f"--> No improvement. Skipping save.\n")

In [321]:
import torch
import torch.nn as nn

file_path = '../data/propaganda_train.tsv'

model_bow = StandardizedClassificationHead(
    input_dim=input_dimension, 
    num_classes=len(LABELS),
    hidden_dim=64,
    dropout_p = 0.3
    )

optimizer = torch.optim.AdamW(model_bow.parameters(), lr=0.0005, weight_decay=0.05)

trainer(
    DATASET_FILE=file_path, 
    LABELS=LABELS, 
    MODEL=model_bow, 
    EPOCHS=5, 
    OPTIMIZER=optimizer
)


--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 1.7040 | Avg Dev Loss: 1.6844

--> New best validation loss achieved, weights saved.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.2072 | Avg Dev Loss: 1.6044

--> New best validation loss achieved, weights saved.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 0.7054 | Avg Dev Loss: 1.6560

--> No improvement. Skipping save.

--- Starting Epoch 4 ---
Epoch 4 Results | Avg Train Loss: 0.4373 | Avg Dev Loss: 1.8369

--> No improvement. Skipping save.

--- Starting Epoch 5 ---
Epoch 5 Results | Avg Train Loss: 0.2944 | Avg Dev Loss: 2.0547

--> No improvement. Skipping save.



In [314]:
import torch
import torch.nn as nn

file_path = '../data/propaganda_train.tsv'

model_bow = StandardizedClassificationHead(
    input_dim=input_dimension, 
    num_classes=len(LABELS),
    hidden_dim=64,
    dropout_p = 0.3
    )

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_bow.parameters(), lr=0.0005, weight_decay=0.05)

label_to_idx = {label: i for i, label in enumerate(LABELS)} # target text to scalar

best_val_loss = float('inf') # early stopping strategy
NUM_EPOCHS = 5

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"--- Starting Epoch {epoch} ---")

    running_train_loss = 0.0
    train_samples = 0
    
    running_val_loss = 0.0
    val_samples = 0


    with open(file_path, mode='r', encoding='utf-8') as file:
        tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
        
        for row_idx, raw_row in enumerate(tsv_reader, start=1):
            label, text = process_row(raw_row)

            if text.count("<BOS>") != 1 or text.count("<EOS>") != 1 or label not in label_to_idx:
                continue

            input_vector = string_to_input_vector(text, word_to_index, pos_to_index, ner_to_index)
            # input_vector_silver = string_to_input_vector(text, word_to_index_silver, pos_to_index, ner_to_index)
        
            x_batched = input_vector
            y_target = torch.tensor([label_to_idx[label]], dtype=torch.long)
            
            if row_idx % 10 == 0:
                # 10% of data routed to validation (Modulo Split)
                model_bow.eval()  # Freeze layers
                with torch.no_grad():
                    logits = model_bow(x_batched)
                    val_loss = criterion(logits, y_target)

                    running_val_loss += val_loss.item()
                    val_samples += 1
            else:
                # 90% of data routed to active training
                model_bow.train()  # Activate layers
                optimizer.zero_grad()
                logits = model_bow(x_batched)
                loss = criterion(logits, y_target)
                loss.backward()
                optimizer.step()

                running_train_loss += loss.item()
                train_samples += 1

    # ========================================================
    # 4d. REPORTING ZONE (File closes, summarize performance)
    # ========================================================
    epoch_train_loss = running_train_loss / train_samples if train_samples > 0 else 0.0
    epoch_val_loss = running_val_loss / val_samples if val_samples > 0 else 0.0
    
    print(f"Epoch {epoch} Results | Avg Train Loss: {epoch_train_loss:.4f} | Avg Dev Loss: {epoch_val_loss:.4f}\n")

    # DYNAMIC EARLY STOPPING SAVE CHECK
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        
        # Save only the parameters (weights), not the whole model class structure
        torch.save(model_bow.state_dict(), 'propaganda_mlp_weights.pt')  # [cite: 1]
        print(f"--> New best validation loss achieved, weights saved.\n")
    else:
        print(f"--> No improvement. Skipping save.\n")


--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 1.7000 | Avg Dev Loss: 1.6524

--> New best validation loss achieved, weights saved.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.1806 | Avg Dev Loss: 1.5787

--> New best validation loss achieved, weights saved.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 0.7044 | Avg Dev Loss: 1.6052

--> No improvement. Skipping save.

--- Starting Epoch 4 ---
Epoch 4 Results | Avg Train Loss: 0.4549 | Avg Dev Loss: 1.7861

--> No improvement. Skipping save.

--- Starting Epoch 5 ---
Epoch 5 Results | Avg Train Loss: 0.3229 | Avg Dev Loss: 1.9445

--> No improvement. Skipping save.



Select the settings where the Dev loss is lowest and before it begins to diverage and rise. This signifies that the model is overfitting and losing its ability to generalize. Continuing to train after the development loss begins to rise degrades performance on unseen data, even if the training loss is falling. The MLP head begins memorizing highly specific keyword co-occurrences rather than learning robust, generalizable propaganda indicators.

This is `EPOCH = 2`. This is also known as early stroppping strategy to capture optimal weights. 

---

IGNORE EVERYTHING BELOW THIS POINT. IT IS NOTES OR DRAFT CODE:

Save the weights:

In [ ]:
# Save only the parameters (weights), not the whole model class structure
torch.save(model_bow.state_dict(), 'propaganda_mlp_weights.pt')
print("Training weights saved safely to disk.")

Pull in the weights:

In [ ]:
# 1. Re-instantiate the empty structural blueprint shell
model_eval = StandardizedClassificationHead(input_dim=input_dimension, num_classes=9)

# 2. Pull the saved weight values back into memory
model_eval.load_state_dict(torch.load('propaganda_mlp_weights.pt', weights_only=True))

# 3. Freeze the model behavior for inference evaluation
model_eval.eval()

# Now, any forward pass run with model_eval uses your loaded parameters

---

In [357]:
import re
import csv
from collections import Counter
import torch
import spacy
from spacy.tokens import Doc
from nltk.tag.perceptron import PerceptronTagger
from nltk.tag import map_tag

class PropagandaFeaturePipeline:
    """
    Encapsulates state (vocabularies, tagsets) while maintaining a pure 
    functional approach to row-by-row string processing and vectorization.
    """
    def __init__(self, spacy_model="en_core_web_sm"):

        self.LABELS = [
            'name_calling,labeling', 'repetition', 'causal_oversimplification', 
            'doubt', 'loaded_language', 'appeal_to_fear_prejudice', 
            'flag_waving', 'exaggeration,minimisation', 'not_propaganda'
        ]
        
        self.UNIVERSAL_TAGSET = ["ADJ","ADP","ADV","CONJ","DET","NOUN","NUM","PRT","PRON","VERB",".","X"]
        
        self.NER_TAG = ['PERSON','ORG','GPE','DATE','NORP','CARDINAL','ORDINAL','TIME','LOC', 'O']
        
        self.CUSTOM_STOPWORDS = ["the" , ",", "to", "of", "and", "in", "a", "that"]

        self.word_to_index = {}
        self.word_to_index_silver = {}
        self.hapax_words_list = []
        self.hapax_words_list_silver = []
        
        self.pos_to_index = self._build_tag_index(self.UNIVERSAL_TAGSET)
        self.ner_to_index = self._build_tag_index(["MISC"] + self.NER_TAG)

        self.nlp = spacy.load(spacy_model)
        self.tagger = PerceptronTagger()

    # ==========================================
    # INTERNAL BUILDER METHODS
    # ==========================================

    def _build_tag_index(self, tagset: list[str]) -> dict[str, int]:
        """Creates a mapping of tags to index positions."""
        return {tag: i for i, tag in enumerate(tagset) if tag not in ["O", "__BOUNDARY__"]}

    
    def build_vocabularies(self, gold_path: str, silver_path: str):
        """
        Parses the datasets to populate the class-level vocabulary matrices.
        This replaces the global counter loops from the notebook.
        """
        global_vocab = Counter()
        global_vocab_silver = Counter()
        
        # 1. Build Gold Vocab
        with open(gold_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in tsv_reader:
                _, text = self.process_row(row)
                global_vocab.update(self.tokenize_whole_words(text))
        
        global_vocab_silver = global_vocab.copy()
                
        # 2. Build Silver Vocab
        with open(silver_path, mode='r', encoding='utf-8') as file:
            tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
            for row in tsv_reader:
                _, text = self.process_row(row)
                tokens = self.tokenize_whole_words(text) 
                
                in_snippet = False # only draw silver counts from synthetic snippet
                for token in tokens:
                    if token == "<BOS>": in_snippet = True; continue
                    if token == "<EOS>": in_snippet = False; continue
                    if in_snippet and token in global_vocab:
                        global_vocab_silver[token] += 1

        # 3. States
        self.hapax_words_list = [word for word, count in global_vocab.items() if count == 1]
        self.hapax_words_list_silver = [word for word, count in global_vocab_silver.items() if count == 1]
        
        gold_list = ["__UNK__"] + [word for word, count in global_vocab.items() if count > 1]
        silver_list = ["__UNK__"] + [word for word, count in global_vocab_silver.items() if count > 1]
        
        self.word_to_index = {w: i for i, w in enumerate(w for w in gold_list if w not in self.CUSTOM_STOPWORDS + ["<EOS>","<BOS>"])}
        self.word_to_index_silver = {w: i for i, w in enumerate(w for w in silver_list if w not in self.CUSTOM_STOPWORDS + ["<EOS>","<BOS>"])}
        print("Vocabulary State Successfully Initialized.")


    # ==========================================
    # FUNCTIONAL TEXT PROCESSING ZONE
    # ==========================================
    def universal_cleaning(self, raw_text: str) -> str:
        """Cleaning directly on raw string format"""
        text = raw_text.strip() # clear leading/trailing whitespace
        text = text.replace("\\'", "'").replace('\\"', '"') # strip out python escape backslashes
        text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'") # standardize quotes to flat quotes
        text = re.sub(r"(?<=\w)'(?=\w)|(?<=[sS])'", '', text) # collapse intra-word apostrophes: won't -> wont, lukes' -> lukes
        text = re.sub(r'[\\/\[\]*|@\ \-.:$#+=]', ' ', text) # remove artifacts: \ / [ ] * | @ space - . : $ # + =
        text = text.replace("<BOS>", " <BOS> ").replace("<EOS>", " <EOS> ") # ensure space around bound tags
        return " ".join(text.split())

    def process_row(self, row: dict) -> tuple[str, str]:
        """Process raw row directly from csv"""
        return row['label'], self.universal_cleaning(row['tagged_in_context'])

    def tokenize_whole_words(self, text: str) -> list[str]:
        """Turn string text into whole-word tokens using regex parser"""
        localized_text = re.sub(r'\b\d+(?:,\d+)*\b', 'num', text)
        pattern = r"<BOS>|<EOS>|(?:[a-zA-Z]\.)+|[a-zA-Z0-9]+(?:[-']?[a-zA-Z0-9]+)*|[^\w\s]"
        raw_tokens = re.findall(pattern, localized_text)
        return [t if t in ["<BOS>", "<EOS>"] else t.lower() for t in raw_tokens]

    def tag_pos_pipeline(self, text: str) -> list[str]:
        """Turn string text into pos tokens using NLTK Perceptron"""
        tokens = self.tokenize_whole_words(text)
        raw_tags = self.tagger.tag(tokens) # nltk PerceptronTagger
        return [
            ("__BOUNDARY__") if t == "<BOS>" or t == "<EOS>" else
            ("NUM") if t.lower() == "num" else # capture num rule from string formatting
            (".") if t in ['"', "'", '`'] else # override mapping
            (map_tag('en-ptb', 'universal', tag)) # map from perceptron native pentree to universal tags
            for t, tag in raw_tags
        ]

    def tag_ner_pipeline(self, text: str) -> list[str]:
        """Turn string text into NER tokens using Spacy"""
        
        allowed = set(self.NER_TAG)

        tokens = self.tokenize_whole_words(text)

        doc = Doc(self.nlp.vocab, words=tokens)
        for name, proc in self.nlp.pipeline: doc = proc(doc)
            
        ner_tags = []
        for token in doc:
            if token.text in ["<BOS>", "<EOS>"]: ner_tags.append("__BOUNDARY__")
            elif token.ent_type_:
                ner_tags.append(f"{token.ent_type_}" if token.ent_type_ in allowed else "MISC")
            else: ner_tags.append("O")
        return ner_tags


    # ==========================================
    # VECTORIZATION ZONE
    # ==========================================
    def string_to_bow_vector(self, string: str, use_silver: bool = False) -> list[int]:
        """Turn text string into a vocab bow python vector"""
        active_vocab = self.word_to_index_silver if use_silver else self.word_to_index
        tokenized = self.tokenize_whole_words(string)
        sequence_vector = [0] * len(active_vocab)

        for token in tokenized:
            # 1. avoiding counting boundaries and stopwords
            if token in ["<EOS>", "<BOS>"] or token in self.CUSTOM_STOPWORDS: continue
            
            # 2. Populate sparse vector
            # get the token from the token, if it doesn't exist, default to __UNK__
            idx = active_vocab.get(token, active_vocab["__UNK__"]) # get 
            sequence_vector[idx] += 1

        return sequence_vector

    def tagset_to_vector(self, string: str, tag_type: str) -> list[int]:
        """Turn text string into a tagset bow python vector"""
        if tag_type == "POS":
            tag_list = self.tag_pos_pipeline(string) 
            active_index = self.pos_to_index
        elif tag_type == "NER": #[cite: 1]
            tag_list = self.tag_ner_pipeline(string)
            active_index = self.ner_to_index

        sequence_vector = [0] * len(active_index)
        for tag in tag_list:
            if tag in ["__BOUNDARY__", "O"]: continue
            sequence_vector[active_index[tag]] += 1
        return sequence_vector

    def string_to_input_vector(self, string_text: str, use_silver: bool = False) -> torch.Tensor:
        """
        Turn text string into a vocab and tagset bow python vector,
        Transform them into PyTorch tensors 
        Concat into single vector for MLP input.
        """
        vocab_vector = self.string_to_bow_vector(string_text, use_silver)
        pos_vector = self.tagset_to_vector(string_text, "POS")
        ner_vector = self.tagset_to_vector(string_text, "NER")

        t_vocab = torch.tensor(vocab_vector, dtype=torch.float32)
        t_pos = torch.tensor(pos_vector, dtype=torch.float32)
        t_ner = torch.tensor(ner_vector, dtype=torch.float32)

        x_combined = torch.cat([t_vocab, t_pos, t_ner], dim=0)
        return x_combined.unsqueeze(0)

---

In [368]:
pipeline = PropagandaFeaturePipeline()

pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv'
)

print(f"List of corpus labels:            {pipeline.LABELS}")
print(f"Universal POS Tagset:             {pipeline.UNIVERSAL_TAGSET}")
print(f"Custom Simplified NER tagset:     {pipeline.NER_TAG}")
print(f"Custom Stopword List:             {pipeline.CUSTOM_STOPWORDS}")
print(f"Unique features in gold:          {len(pipeline.word_to_index)}")
print(f"Unique features in gold + silver: {len(pipeline.word_to_index_silver)}")
print(f"Dims of baseline gold sparse vec: {len(pipeline.hapax_words_list)}")
print(f"Dims of gold + silver sparse vec: {len(pipeline.hapax_words_list_silver)}")
print(f"Dims of POS vector: {len(pipeline.pos_to_index)}")
print(f"Dimes of NER vector: {len(pipeline.ner_to_index)}")

print(f"POS Tagger: {pipeline.tagger}")
print(f"NER Tagger: {pipeline.nlp}")


Vocabulary State Successfully Initialized.
List of corpus labels:            ['name_calling,labeling', 'repetition', 'causal_oversimplification', 'doubt', 'loaded_language', 'appeal_to_fear_prejudice', 'flag_waving', 'exaggeration,minimisation', 'not_propaganda']
Universal POS Tagset:             ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRT', 'PRON', 'VERB', '.', 'X']
Custom Simplified NER tagset:     ['PERSON', 'ORG', 'GPE', 'DATE', 'NORP', 'CARDINAL', 'ORDINAL', 'TIME', 'LOC', 'O']
Custom Stopword List:             ['the', ',', 'to', 'of', 'and', 'in', 'a', 'that']
Unique features in gold:          4618
Unique features in gold + silver: 5246
Dims of baseline gold sparse vec: 4037
Dims of gold + silver sparse vec: 3409
Dims of POS vector: 12
Dimes of NER vector: 10
POS Tagger: <nltk.tag.perceptron.PerceptronTagger object at 0x16a409180>
NER Tagger: <spacy.lang.en.English object at 0x30ff8ba00>


In [369]:
import csv
import torch
import torch.nn as nn

class PropagandaTrainer:
    """
    All-in-one manager that builds the standardized PyTorch MLP architecture,
    configures optimization, and executes the streaming training loop.
    """
    def __init__(
        self, 
        pipeline, 
        hidden_dim: int = 64, 
        dropout_p: float = 0.3,
        lr: float = 0.0005,
        weight_decay: float = 0.05,
        use_silver: bool = False
    ):
        self.pipeline = pipeline
        self.use_silver = use_silver
        self.label_to_idx = {label: i for i, label in enumerate(pipeline.LABELS)}
        self.best_val_loss = float('inf')
        
        # 1. Dynamically compute total input dimension from pipeline state
        active_vocab = pipeline.word_to_index_silver if use_silver else pipeline.word_to_index
        input_dimension = (
            len(active_vocab) + 
            len(pipeline.pos_to_index) + 
            len(pipeline.ner_to_index)
        )
        
        # 2. Build and hold classification head
        self.model = self._build_head(
            input_dim=input_dimension,
            hidden_dim=hidden_dim,
            num_classes=len(pipeline.LABELS),
            dropout_p=dropout_p
        )
        
        # 3. Configure head components
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(), 
            lr=lr, 
            weight_decay=weight_decay
        )

    # ==========================================
    # NETWORK BUILDER
    # ==========================================
    def _build_head(self, input_dim: int, hidden_dim: int, num_classes: int, dropout_p: float) -> nn.Module:
        """
        Constructs the standardized MLP classification head directly.
        Uses LayerNorm instead of BatchNorm1d to ensure stability during batch_size=1 streaming.
        """
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),  # Safe for single-instance streaming
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, num_classes)
        )

    # ==========================================
    # STREAMING TRAINING PASS
    # ==========================================
    def run_training_loop(
        self, 
        dataset_path: str, 
        epochs: int = 5, 
        save_path: str = 'propaganda_mlp_weights.pt'
    ):
        """
        Executes row-by-row streaming training and 10% modulo validation.
        """

        for epoch in range(1, epochs + 1):
            print(f"--- Starting Epoch {epoch} ---")

            running_train_loss, train_samples = 0.0, 0
            running_val_loss, val_samples = 0.0, 0

            with open(dataset_path, mode='r', encoding='utf-8') as file:
                tsv_reader = csv.DictReader(file, delimiter='\t', quoting=csv.QUOTE_NONE)
                
                for row_idx, raw_row in enumerate(tsv_reader, start=1):
                    label, text = self.pipeline.process_row(raw_row)

                    # Formatting guardrail
                    if text.count("<BOS>") != 1 or text.count("<EOS>") != 1 or label not in self.label_to_idx:
                        print(f"Warning: Input data error, {row_idx}, {raw_row}")
                        continue

                    # Feature Extraction
                    x_batched = self.pipeline.string_to_input_vector(text, use_silver=self.use_silver)
                    y_target = torch.tensor([self.label_to_idx[label]], dtype=torch.long)
                    
                    # 10% Modulo Split for internal dev validation
                    if row_idx % 10 == 0:
                        self.model.eval() # testing
                        with torch.no_grad():
                            logits = self.model(x_batched)
                            val_loss = self.criterion(logits, y_target)
                            running_val_loss += val_loss.item()
                            val_samples += 1
                    else:
                        self.model.train() # training
                        self.optimizer.zero_grad()
                        logits = self.model(x_batched)
                        loss = self.criterion(logits, y_target)
                        loss.backward()
                        self.optimizer.step()
                        
                        running_train_loss += loss.item()
                        train_samples += 1

            # Epoch reporting
            epoch_train_loss = running_train_loss / train_samples if train_samples > 0 else 0.0
            epoch_val_loss = running_val_loss / val_samples if val_samples > 0 else 0.0
            
            print(f"Epoch {epoch} Results | Avg Train Loss: {epoch_train_loss:.4f} | Avg Dev Loss: {epoch_val_loss:.4f}")

            # Early stopping checkpoint saving
            if epoch_val_loss < self.best_val_loss:
                self.best_val_loss = epoch_val_loss
                torch.save(self.model.state_dict(), save_path)
                print(f"--> New best validation loss ({self.best_val_loss:.4f}) achieved! Model saved to {save_path}.\n")
            else:
                print(f"--> No improvement on validation loss. Skipping save.\n")

In [370]:
# 1. Initialize and build feature pipeline state
pipeline = PropagandaFeaturePipeline()
pipeline.build_vocabularies(
    gold_path='../data/propaganda_train.tsv', 
    silver_path='../data/silver_train.tsv'
)

# 2. Instantiate trainer (automatically sets up model dimensions & AdamW)
trainer = PropagandaTrainer(
    pipeline=pipeline,
    hidden_dim=64,
    dropout_p=0.3,
    lr=0.0005,
    weight_decay=0.05,
    use_silver=False
)

# 3. Launch dynamic training pass
trainer.run_training_loop(
    dataset_path='../data/propaganda_train_100.tsv',
    epochs=5,
    save_path='propaganda_mlp_weights.pt'
)

Vocabulary State Successfully Initialized.
--- Starting Epoch 1 ---
Epoch 1 Results | Avg Train Loss: 1.8085 | Avg Dev Loss: 2.0443
--> New best validation loss (2.0443) achieved! Model saved to propaganda_mlp_weights.pt.

--- Starting Epoch 2 ---
Epoch 2 Results | Avg Train Loss: 1.3101 | Avg Dev Loss: 1.9846
--> New best validation loss (1.9846) achieved! Model saved to propaganda_mlp_weights.pt.

--- Starting Epoch 3 ---
Epoch 3 Results | Avg Train Loss: 0.7022 | Avg Dev Loss: 1.9986
--> No improvement on validation loss. Skipping save.

--- Starting Epoch 4 ---
Epoch 4 Results | Avg Train Loss: 0.3936 | Avg Dev Loss: 2.0192
--> No improvement on validation loss. Skipping save.

--- Starting Epoch 5 ---
Epoch 5 Results | Avg Train Loss: 0.2559 | Avg Dev Loss: 2.0474
--> No improvement on validation loss. Skipping save.

